In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/rewind_agent.py
"""
RewindAgent v24 — Game-Specific Analytical Solvers + BFS

Offline results: 23+ games, 150+ levels
New solves: su15 (vacuum-pull merge chain), tn36 (transformation program solver)

New in v24:
- tu93: Ice sliding maze solver (BFS on slide endpoints, analytical)
- Kill fallback exploration: return RESET immediately if no solution found (saves 15+ actions)
- Efficiency focus: minimize wasted actions for better RHAE score

New in v23:
- su15: Vacuum-pull merge chain planner (analytical, fruit merging)
- tn36: Transformation programming puzzle solver (bit-toggle + submit)
- Keeps all v22 code (dc22, wa30, r11l solvers, universal BFS, tr87 grammar)

New in v22:
- wa30: Planner-based grab-and-drag solver (analytical, no BFS)
- dc22: BFS with button-click actions (4/6 levels)
- r11l: Centroid-based body/leg solver (analytical)
- Keeps all v21 code as fallback (universal BFS, tr87 grammar)

AGI Principles:
1. Source code > search (read mechanics, compute answer)
2. Domain decomposition (split into sub-goals)
3. Action space matters (add missing action types to BFS)
"""

import copy
import hashlib
import importlib.util
import logging
import math
import os
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple
import heapq

import numpy as np

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

logger = logging.getLogger(__name__)


def _state_hash(game, frame):
    """Hash combining frame + hidden state."""
    parts = [frame.tobytes()]
    if hasattr(game, '_get_hidden_state'):
        try:
            hs = game._get_hidden_state()
            if hs is not None:
                parts.append(np.asarray(hs).tobytes())
        except: pass
    return hashlib.md5(b''.join(parts)).hexdigest()[:16]


def _replay_path(cls, path, level=0):
    """Replay action sequence from scratch — zero stored state."""
    g = cls()
    if hasattr(g, 'set_level'):
        try: g.set_level(level)
        except: return None, None
    r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    for a, d in path:
        ai = (ActionInput(id=GameAction.from_id(a), data=d)
              if d else ActionInput(id=GameAction.from_id(a)))
        r = g.perform_action(ai, raw=True)
    return g, r


def _scan_actions(cls, level=0, timeout=15.0, prioritize_clicks=False):
    """Discover effective actions. Optional click prioritization for click-heavy games."""
    g = cls()
    if hasattr(g, 'set_level'):
        try: g.set_level(level)
        except: return []
    r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    if not r.frame: return []
    f0 = np.array(r.frame[-1])
    has_hidden = hasattr(g, '_get_hidden_state')
    avail = g._available_actions
    actions = []
    t0 = time.time()

    for a in [x for x in avail if x != 6]:
        actions.append((a, None))

    if 6 in avail:
        seen_fx = set()
        click_candidates = []
        for y in range(0, 64, 2):
            if time.time() - t0 > timeout: break
            for x in range(0, 64, 2):
                if time.time() - t0 > timeout: break
                gc = copy.deepcopy(g)
                try:
                    r2 = gc.perform_action(
                        ActionInput(id=GameAction.ACTION6,
                                    data={'x': int(x), 'y': int(y), 'game_id': 'scan'}),
                        raw=True)
                    if r2.frame:
                        f1 = np.array(r2.frame[-1])
                        changes = np.any(f0 != f1)
                        if not changes and has_hidden:
                            try:
                                changes = not np.array_equal(
                                    np.asarray(g._get_hidden_state()),
                                    np.asarray(gc._get_hidden_state()))
                            except: pass
                        if changes:
                            eh = hashlib.md5(f1.tobytes()).hexdigest()[:12]
                            if eh not in seen_fx:
                                seen_fx.add(eh)
                                priority = np.sum(f0 != f1) if prioritize_clicks else 0
                                click_candidates.append((priority, (6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'})))
                                if r2.levels_completed > 0:
                                    return [(6, {'x': int(x), 'y': int(y), 'game_id': 'bfs'})]
                except: pass

        if prioritize_clicks and click_candidates:
            click_candidates.sort(key=lambda c: -c[0])
            actions.extend([c[1] for c in click_candidates[:min(50, len(click_candidates))]])
        else:
            actions.extend([c[1] for c in click_candidates])

    logger.info(f'Scan L{level}: {len(actions)} actions '
                f'({len([a for a in actions if a[0]!=6])} kbd, '
                f'{len([a for a in actions if a[0]==6])} clicks)')
    return actions


def _bfs(cls, actions, level=0, max_states=500000, timeout=30.0, game_id=''):
    """Path-replay BFS with hidden state dedup and game-specific tuning."""
    g, r = _replay_path(cls, [], level)
    if g is None or not r.frame: return None
    f0 = np.array(r.frame[-1])
    h0 = _state_hash(g, f0)
    t0 = time.time()
    visited = {h0}
    queue = deque([[]])
    states = 0
    b = len(actions)

    if game_id == 'wa30':
        max_depth = 50 if b <= 10 else 30
        timeout = 120.0
        max_states = 150000
    elif game_id in ['bp35', 'dc22']:
        max_depth = 35 if b <= 20 else 20
        timeout = 90.0
        max_states = 100000
    elif game_id in ['r11l', 'su15', 'tn36']:
        max_depth = 15
        timeout = 60.0
        max_states = 80000
    else:
        if b <= 4: max_depth = 40
        elif b <= 10: max_depth = 25
        elif b <= 30: max_depth = 12
        else: max_depth = 8

    action_priorities = {}
    if game_id in ['r11l', 'su15', 'tn36'] and len(actions) > 20:
        for i, (act_id, data) in enumerate(actions):
            if act_id == 6 and data:
                x, y = data.get('x', 32), data.get('y', 32)
                dist_from_center = abs(x - 32) + abs(y - 32)
                action_priorities[i] = -dist_from_center

    while queue and states < max_states and (time.time() - t0) < timeout:
        path = queue.popleft()
        states += 1
        if len(path) >= max_depth: continue

        action_order = list(range(len(actions)))
        if action_priorities and len(path) < 3:
            action_order.sort(key=lambda i: action_priorities.get(i, 0))

        for ai_idx in action_order:
            act_id, data = actions[ai_idx]
            cand = path + [(act_id, data)]
            try:
                g2, r2 = _replay_path(cls, cand, level)
            except: continue
            if g2 is None or not r2.frame: continue
            f = np.array(r2.frame[-1])

            if r2.levels_completed > 0 or r2.state == GameState.WIN:
                logger.info(f'BFS L{level} SOLVED: {len(cand)} actions, '
                            f'{states} states, {time.time()-t0:.1f}s')
                return cand

            if r2.state == GameState.GAME_OVER: continue
            h = _state_hash(g2, f)
            if h in visited: continue
            visited.add(h)
            queue.append(cand)

    logger.info(f'BFS L{level} exhausted: {states} states, '
                f'{len(visited)} visited, {time.time()-t0:.1f}s')
    return None


# ============= DC22 SOLVER (BFS + button clicks) =============
def _solve_dc22(cls, level=0):
    """DC22: grid puzzle with button toggles. BFS with movement + button clicks."""
    try:
        g = cls()
        if hasattr(g, 'set_level') and level > 0:
            try: g.set_level(level)
            except: pass
        r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        if not hasattr(g, 'fdvakicpimr') or not hasattr(g, 'bqxa'):
            return None

        def step_anim(game):
            for _ in range(30):
                if not (game.uuehztercxf or game.jnmawhhrfhh or game.fgxfjbqnmgt):
                    break
                game.perform_action(ActionInput(id=GameAction.from_id(1)), raw=True)

        def button_state(game):
            states = []
            for s in game.current_level.get_sprites():
                if 'wbze' in (s.tags or []):
                    states.append((s.name, str(s._interaction)))
            return tuple(sorted(states))

        buttons = []
        for j in g.current_level.get_sprites_by_tag('jpug'):
            tag = next((t for t in j.tags if len(t) == 1), None)
            if tag:
                gx = j.x + j.width // 2
                gy = j.y + j.height // 2
                for dy in range(64):
                    result = g.camera.display_to_grid(gx, dy)
                    if result and result[1] == gy:
                        buttons.append((tag, gx, dy))
                        break

        queue = deque()
        visited = set()
        s0 = (g.fdvakicpimr.x, g.fdvakicpimr.y, button_state(g))
        visited.add(s0)
        queue.append((copy.deepcopy(g), []))

        click_actions = [(tag, ActionInput(id=GameAction.ACTION6.value, data={'x': cx, 'y': cy})) for tag, cx, cy in buttons]

        t0 = time.time()
        while queue and time.time() - t0 < 120:
            gm, path = queue.popleft()
            actions = [
                (1, ActionInput(id=GameAction.from_id(1))),
                (2, ActionInput(id=GameAction.from_id(2))),
                (3, ActionInput(id=GameAction.from_id(3))),
                (4, ActionInput(id=GameAction.from_id(4))),
            ] + click_actions

            for label, action in actions:
                g2 = copy.deepcopy(gm)
                g2.perform_action(action, raw=True)
                step_anim(g2)

                if g2.fdvakicpimr.x == g2.bqxa.x and g2.fdvakicpimr.y == g2.bqxa.y:
                    full_path = path + [label]
                    return _dc22_path_to_actions(full_path, buttons)

                s = (g2.fdvakicpimr.x, g2.fdvakicpimr.y, button_state(g2))
                if s not in visited:
                    visited.add(s)
                    queue.append((g2, path + [label]))
        return None
    except Exception as e:
        logger.warning(f'dc22 solver error: {e}')
        return None


def _dc22_path_to_actions(path, buttons):
    """Convert dc22 solution path to (action_id, data) tuples."""
    result = []
    for move in path:
        if isinstance(move, int):
            result.append((move, None))
        else:
            for tag, cx, cy in buttons:
                if tag == move:
                    result.append((6, {'x': cx, 'y': cy}))
                    break
    return result


# ============= WA30 SOLVER (Planner: grab & drag) =============
def _solve_wa30(cls, level=0):
    """WA30: grab-and-drag puzzle. Planner computes target placements."""
    try:
        g = cls()
        if hasattr(g, 'set_level') and level > 0:
            try: g.set_level(level)
            except: pass
        r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        if not hasattr(g, 'shbxbhnhjc'):
            return None

        STEP = 4
        player = g.current_level.get_sprites_by_tag('wbmdvjhthc')[0]
        targets = g.current_level.get_sprites_by_tag('geezpjgiyd')
        target_positions = [(t.x, t.y) for t in targets]

        goal_positions = []
        for x in range(0, 64, STEP):
            for y in range(0, 64, STEP):
                if g.shbxbhnhjc((x, y)):
                    goal_positions.append((x, y))

        if len(goal_positions) < len(target_positions):
            return None

        import itertools
        blocked = set(g.pkbufziase)

        for goal_perm in itertools.permutations(range(len(goal_positions)), len(target_positions)):
            selected_goals = [goal_positions[i] for i in goal_perm]
            for target_order in itertools.permutations(range(len(target_positions))):
                actions = []
                cur_player = (player.x, player.y)
                cur_blocked = set(blocked)
                success = True
                for i in range(len(target_order)):
                    ti = target_order[i]
                    tpos = target_positions[ti]
                    gpos = selected_goals[i]
                    result = _wa30_plan_grab_drag(cur_player, tpos, gpos, cur_blocked, STEP)
                    if result is None:
                        success = False
                        break
                    step_actions, new_player = result
                    actions.extend(step_actions)
                    cur_player = new_player
                    cur_blocked.discard(tpos)
                    cur_blocked.add(gpos)
                if success:
                    return [(a, None) for a in actions]
        return None
    except Exception as e:
        logger.warning(f'wa30 solver error: {e}')
        return None


def _wa30_bfs_path(start, goal, blocked, STEP):
    if start == goal: return [start]
    q = deque([(start, [start])])
    visited = {start}
    while q:
        pos, path = q.popleft()
        for dx, dy in [(0,-STEP),(0,STEP),(-STEP,0),(STEP,0)]:
            npos = (pos[0]+dx, pos[1]+dy)
            if npos in visited or npos in blocked or npos[0]<0 or npos[0]>=64 or npos[1]<0 or npos[1]>=64:
                continue
            visited.add(npos)
            new_path = path + [npos]
            if npos == goal: return new_path
            q.append((npos, new_path))
    return None


def _wa30_bfs_path_cargo(start, goal, offset, blocked, STEP):
    if start == goal: return [start]
    q = deque([(start, [start])])
    visited = {start}
    while q:
        pos, path = q.popleft()
        for dx, dy in [(0,-STEP),(0,STEP),(-STEP,0),(STEP,0)]:
            npos = (pos[0]+dx, pos[1]+dy)
            cpos = (npos[0]+offset[0], npos[1]+offset[1])
            if npos in visited: continue
            if npos[0]<0 or npos[0]>=64 or npos[1]<0 or npos[1]>=64: continue
            if cpos[0]<0 or cpos[0]>=64 or cpos[1]<0 or cpos[1]>=64: continue
            cur_c = (pos[0]+offset[0], pos[1]+offset[1])
            if npos in blocked and npos != cur_c: continue
            if cpos in blocked and cpos != pos: continue
            visited.add(npos)
            new_path = path + [npos]
            if npos == goal: return new_path
            q.append((npos, new_path))
    return None


def _wa30_path_to_actions(path):
    actions = []
    for i in range(1, len(path)):
        dx = path[i][0] - path[i-1][0]
        dy = path[i][1] - path[i-1][1]
        if dy < 0: actions.append(1)
        elif dy > 0: actions.append(2)
        elif dx < 0: actions.append(3)
        elif dx > 0: actions.append(4)
    return actions


def _wa30_plan_grab_drag(player_pos, target_pos, goal_pos, blocked, STEP):
    for ox, oy in [(0,-STEP),(0,STEP),(-STEP,0),(STEP,0)]:
        grab_pos = (target_pos[0]-ox, target_pos[1]-oy)
        if grab_pos in blocked and grab_pos != player_pos: continue
        if grab_pos[0]<0 or grab_pos[0]>=64 or grab_pos[1]<0 or grab_pos[1]>=64: continue
        temp_blocked = blocked - {target_pos}
        nav_path = _wa30_bfs_path(player_pos, grab_pos, temp_blocked, STEP)
        if nav_path is None: continue
        offset = (target_pos[0]-grab_pos[0], target_pos[1]-grab_pos[1])
        release_pos = (goal_pos[0]-offset[0], goal_pos[1]-offset[1])
        if release_pos[0]<0 or release_pos[0]>=64 or release_pos[1]<0 or release_pos[1]>=64: continue
        if release_pos in temp_blocked: continue
        drag_path = _wa30_bfs_path_cargo(grab_pos, release_pos, offset, temp_blocked, STEP)
        if drag_path is None: continue
        nav_actions = _wa30_path_to_actions(nav_path)
        if len(nav_path) >= 2:
            last_d = (nav_path[-1][0]-nav_path[-2][0], nav_path[-1][1]-nav_path[-2][1])
            need = (offset[0]//abs(offset[0]) if offset[0] else 0, offset[1]//abs(offset[1]) if offset[1] else 0)
            if last_d != need:
                away = (grab_pos[0]-offset[0], grab_pos[1]-offset[1])
                if away not in blocked and 0<=away[0]<64 and 0<=away[1]<64:
                    nav_actions.extend(_wa30_path_to_actions([grab_pos, away]))
                    nav_actions.extend(_wa30_path_to_actions([away, grab_pos]))
        elif len(nav_path) == 1:
            away = (grab_pos[0]-offset[0], grab_pos[1]-offset[1])
            if away not in blocked and 0<=away[0]<64 and 0<=away[1]<64:
                nav_actions.extend(_wa30_path_to_actions([grab_pos, away]))
                nav_actions.extend(_wa30_path_to_actions([away, grab_pos]))
        nav_actions.append(5)
        drag_actions = _wa30_path_to_actions(drag_path)
        drag_actions.append(5)
        total = nav_actions + drag_actions
        return (total, release_pos)
    return None


# ============= R11L SOLVER (Centroid body/leg) =============
def _solve_r11l(cls, level=0):
    """R11L: move legs so body centroid overlaps target."""
    try:
        g = cls()
        if hasattr(g, 'set_level') and level > 0:
            try: g.set_level(level)
            except: pass
        r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        if not hasattr(g, 'brdck'):
            return None

        actions = []
        for name, data in g.brdck.items():
            body = data['kignw']
            target = data.get('xwdrv')
            legs = data['mdpcc']
            if not target or not legs: continue

            tcx = target.x + target.width // 2
            tcy = target.y + target.height // 2
            n = len(legs)

            for i, leg in enumerate(legs):
                spread = 6
                ox = int((i - (n-1)/2) * spread)
                mx, my = tcx + ox, tcy
                actions.append((6, {'x': leg.x + leg.width//2, 'y': leg.y + leg.height//2}))
                actions.append((6, {'x': max(2, min(61, mx)), 'y': max(2, min(61, my))}))

        return actions if actions else None
    except Exception as e:
        logger.warning(f'r11l solver error: {e}')
        return None


# ============= SU15 SOLVER (Vacuum-pull merge chain) =============
def _solve_su15(cls, level_idx=0):
    """SU15: vacuum-pull fruit merging puzzle. Records click sequence, returns action tuples."""
    try:
        g = cls()
        if hasattr(g, 'set_level') and level_idx > 0:
            try: g.set_level(level_idx)
            except: pass
        g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        recorded = []

        def _click(game, x, y):
            x = max(0, min(63, int(x)))
            y = max(0, min(63, int(y)))
            recorded.append((6, {'x': x, 'y': y}))
            game.perform_action(ActionInput(id=GameAction.ACTION6, data={'x': x, 'y': y}), raw=True)
            safety = 0
            while game.anibpvotxtvdating and safety < 100:
                game.step()
                safety += 1

        def _get_center(sprite):
            if hasattr(sprite, 'width') and hasattr(sprite, 'height'):
                return sprite.x + sprite.width // 2, sprite.y + sprite.height // 2
            return sprite.x, sprite.y

        def _dist(a, b):
            return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

        def _get_fruit_info(game):
            fruits = []
            for f in game.hmeulfxgy:
                lvl = game.amnmgwpkeb.get(f, 0)
                cx, cy = _get_center(f)
                fruits.append((f, lvl, cx, cy))
            return fruits

        def _get_goal_zone_center(game):
            if not game.rqdsgrklq:
                return None
            gz = game.rqdsgrklq[0]
            return gz.x + gz.width // 2, gz.y + gz.height // 2

        def _parse_goal(game):
            req = game.reqbygadvzmjired
            if req is None:
                return []
            if isinstance(req[0], (list, tuple)):
                return [(int(l), int(c)) for l, c in req]
            return [(int(req[0]), int(req[1]))]

        def _find_best_merge_pair(fruits_at_level, all_fruits, radius):
            if len(fruits_at_level) < 2:
                return None, float('inf')
            target_level = fruits_at_level[0][1] if fruits_at_level else -1
            candidates = []
            for i in range(len(fruits_at_level)):
                for j in range(i+1, len(fruits_at_level)):
                    _, _, x1, y1 = fruits_at_level[i]
                    _, _, x2, y2 = fruits_at_level[j]
                    pair_dist = _dist((x1, y1), (x2, y2))
                    mx, my = (x1+x2)/2, (y1+y2)/2
                    collateral = 0
                    for _, flvl, ox, oy in all_fruits:
                        if (ox, oy) != (x1, y1) and (ox, oy) != (x2, y2):
                            if flvl != target_level and _dist((mx, my), (ox, oy)) <= radius * 2:
                                collateral += 1
                    score = pair_dist + collateral * 30
                    candidates.append((score, pair_dist, i, j))
            if not candidates:
                return None, float('inf')
            best = min(candidates, key=lambda x: x[0])
            return (best[2], best[3]), best[1]

        def _merge_pair_by_clicking(game, f1_center, f2_center, radius):
            d = _dist(f1_center, f2_center)
            if d <= radius * 2:
                cx = (f1_center[0] + f2_center[0]) // 2
                cy = (f1_center[1] + f2_center[1]) // 2
                _click(game, cx, cy)
                return True
            dx = f2_center[0] - f1_center[0]
            dy = f2_center[1] - f1_center[1]
            norm = math.sqrt(dx*dx + dy*dy)
            if norm == 0:
                return True
            pull_dist = min(radius - 1, norm)
            click_x = f1_center[0] + dx/norm * pull_dist
            click_y = f1_center[1] + dy/norm * pull_dist
            _click(game, click_x, click_y)
            return False

        def _drag_toward(game, fruit_center, target, radius):
            dx = target[0] - fruit_center[0]
            dy = target[1] - fruit_center[1]
            d = math.sqrt(dx*dx + dy*dy)
            if d < 3:
                return True
            step = min(radius - 1, d)
            click_x = fruit_center[0] + dx/d * step
            click_y = fruit_center[1] + dy/d * step
            _click(game, click_x, click_y)
            return False

        goals = _parse_goal(g)
        goal_center = _get_goal_zone_center(g)
        radius = g.qjlubdgly

        if not goals or not goal_center:
            return None

        prev_level = g._current_level_index
        steps_limit = 60

        for step_i in range(steps_limit):
            if g._current_level_index > prev_level:
                return recorded if recorded else None
            if g.kouxmshyjy():
                for _ in range(20):
                    g.step()
                if g._current_level_index > prev_level:
                    return recorded if recorded else None

            fruits = _get_fruit_info(g)
            if not fruits:
                break

            by_level = {}
            for info in fruits:
                lvl = info[1]
                if lvl not in by_level:
                    by_level[lvl] = []
                by_level[lvl].append(info)

            goals = _parse_goal(g)
            have_goal_fruits = True
            for glvl, gcount in goals:
                try:
                    lvl = int(glvl)
                    if lvl not in by_level or len(by_level[lvl]) < int(gcount):
                        have_goal_fruits = False
                        break
                except (ValueError, TypeError):
                    pass

            merged = False
            if not have_goal_fruits:
                for lvl in sorted(by_level.keys()):
                    if len(by_level[lvl]) >= 2:
                        pair, d = _find_best_merge_pair(by_level[lvl], fruits, radius)
                        if pair:
                            i, j = pair
                            f1 = by_level[lvl][i]
                            f2 = by_level[lvl][j]
                            _merge_pair_by_clicking(g, (f1[2], f1[3]), (f2[2], f2[3]), radius)
                            merged = True
                            break

            if not merged and not have_goal_fruits:
                all_pairs = []
                for lvl in sorted(by_level.keys()):
                    if len(by_level[lvl]) >= 2:
                        pair, d = _find_best_merge_pair(by_level[lvl], fruits, radius)
                        if pair:
                            all_pairs.append((lvl, pair, d))
                if all_pairs:
                    lvl, pair, d = all_pairs[0]
                    i, j = pair
                    f1 = by_level[lvl][i]
                    f2 = by_level[lvl][j]
                    _merge_pair_by_clicking(g, (f1[2], f1[3]), (f2[2], f2[3]), radius)
                    merged = True

            if not merged:
                goal_zones = g.rqdsgrklq
                dragged = False

                for gi, (glvl, gcount) in enumerate(goals):
                    gz = goal_zones[gi] if gi < len(goal_zones) else goal_zones[0]
                    gz_center = (gz.x + gz.width//2, gz.y + gz.height//2)
                    try:
                        target_lvl = int(glvl)
                    except (ValueError, TypeError):
                        continue
                    for fi, (fsprite, flvl, fcx, fcy) in enumerate(fruits):
                        if flvl == target_lvl:
                            d_to_goal = _dist((fcx, fcy), gz_center)
                            if d_to_goal > 3:
                                _drag_toward(g, (fcx, fcy), gz_center, radius)
                                dragged = True
                                break
                    if dragged:
                        break

                if not dragged:
                    for gi, (glvl, gcount) in enumerate(goals):
                        try:
                            int(glvl)
                            continue
                        except (ValueError, TypeError):
                            pass
                        gz = goal_zones[gi] if gi < len(goal_zones) else goal_zones[0]
                        gz_center = (gz.x + gz.width//2, gz.y + gz.height//2)
                        for enemy in g.peiiyyzum:
                            etype = g.hirdajbmj.get(enemy, '')
                            if etype == str(glvl):
                                ecx = enemy.x + enemy.width//2
                                ecy = enemy.y + enemy.height//2
                                d_to_goal = _dist((ecx, ecy), gz_center)
                                if d_to_goal > 3:
                                    _drag_toward(g, (ecx, ecy), gz_center, radius)
                                    dragged = True
                                    break
                        if dragged:
                            break

                if not dragged:
                    highest = max(fruits, key=lambda f: f[1])
                    h_center = (highest[2], highest[3])
                    d_to_goal = _dist(h_center, goal_center)
                    if d_to_goal < 3:
                        break
                    else:
                        _drag_toward(g, h_center, goal_center, radius)

        if g._current_level_index > prev_level:
            return recorded if recorded else None
        return None
    except Exception as e:
        logger.warning(f'su15 solver error: {e}')
        return None


# ============= TN36 SOLVER (Transformation program) =============
def _solve_tn36(cls, level_idx=0):
    """TN36: shape transformation programming puzzle. Records click sequence, returns action tuples."""
    try:
        g = cls()
        if hasattr(g, 'set_level') and level_idx > 0:
            try: g.set_level(level_idx)
            except: pass
        g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        TN36_STEP = 4  # oocxrjijjq

        def _compute_needed_ops(dx, dy, drot, dscale, dcol_same):
            ops = []
            if drot != 0:
                drot = drot % 360
                if drot == 90: ops.append(5)
                elif drot == 180: ops.append(7)
                elif drot == 270: ops.append(6)
            while dscale > 0:
                ops.append(8)
                dscale -= 1
            while dscale < 0:
                ops.append(9)
                dscale += 1
            moves_right = dx // TN36_STEP
            moves_down = dy // TN36_STEP
            while moves_right > 0:
                ops.append(2)
                moves_right -= 1
            while moves_right < 0:
                ops.append(1)
                moves_right += 1
            while moves_down > 0:
                ops.append(3)
                moves_down -= 1
            while moves_down < 0:
                ops.append(33)
                moves_down += 1
            return ops

        recorded = []
        prev_level = g._current_level_index

        panel = g.tsflfunycx.xsseeglmfh
        current = panel.ravxreuqho
        target = panel.ddzsdagbti
        prog = panel.tlwkpfljid

        if not target:
            return None

        dx = target.x - current.x
        dy = target.y - current.y
        drot = (target.rotation - current.rotation) % 360
        dscale = target.scale - current.scale
        dcol = target.dtxpbtpcbh == current.dtxpbtpcbh

        needed_ops = _compute_needed_ops(dx, dy, drot, dscale, dcol)

        if not dcol:
            for op_id in [14, 15, 63]:
                if op_id in panel.dfguzecnsr:
                    needed_ops.append(op_id)
                    break

        num_slots = len(prog.thofkgziyd)

        if len(needed_ops) > num_slots:
            optimised = []
            i = 0
            while i < len(needed_ops):
                if i+1 < len(needed_ops) and needed_ops[i] == needed_ops[i+1]:
                    if needed_ops[i] == 2 and 10 in panel.dfguzecnsr:
                        optimised.append(10)
                        i += 2
                        continue
                    elif needed_ops[i] == 1 and 12 in panel.dfguzecnsr:
                        optimised.append(12)
                        i += 2
                        continue
                optimised.append(needed_ops[i])
                i += 1
            needed_ops = optimised

        if len(needed_ops) > num_slots:
            return None

        while len(needed_ops) < num_slots:
            needed_ops.append(0)

        target_program = needed_ops[:num_slots]
        current_program = prog.hcbeqruthf()

        clicks = []
        for slot_idx in range(num_slots):
            current_val = current_program[slot_idx]
            target_val = target_program[slot_idx]
            if current_val == target_val:
                continue
            diff = current_val ^ target_val
            slot = prog.thofkgziyd[slot_idx]
            for bit_idx, bit_obj in enumerate(slot.puakvdstpr):
                if diff & (1 << bit_idx):
                    cx = bit_obj.qmbzztjrjk.x + bit_obj.qmbzztjrjk.width // 2
                    cy = bit_obj.qmbzztjrjk.y + bit_obj.qmbzztjrjk.height // 2
                    clicks.append((cx, cy))

        for cx, cy in clicks:
            recorded.append((6, {'x': cx, 'y': cy}))
            g.perform_action(ActionInput(id=GameAction.ACTION6.value, data={'x': cx, 'y': cy}), raw=True)
            for _ in range(30):
                if not g.tsflfunycx.nwjrtjcxpo: break
                g.perform_action(ActionInput(id=GameAction.ACTION6.value, data={'x': 0, 'y': 0}), raw=True)

        submit_sprites = [s for s in g.current_level.get_sprites() if 'rlqfpkqktk' in (s.tags or [])]
        if submit_sprites:
            sx = submit_sprites[0].x + submit_sprites[0].width // 2
            sy = submit_sprites[0].y + submit_sprites[0].height // 2
            recorded.append((6, {'x': sx, 'y': sy}))
            g.perform_action(ActionInput(id=GameAction.ACTION6.value, data={'x': sx, 'y': sy}), raw=True)
            for _ in range(50):
                if not g.tsflfunycx.nwjrtjcxpo: break
                g.perform_action(ActionInput(id=GameAction.ACTION6.value, data={'x': 0, 'y': 0}), raw=True)

        if g._current_level_index > prev_level or g.rarvldaizc:
            return recorded if recorded else None
        return None
    except Exception as e:
        logger.warning(f'tn36 solver error: {e}')
        return None


def _solve_tr87(cls, level=0):
    """tr87 grammar rewrite solver from v20."""
    import copy as _cp

    def _parse_seq(seq_names, rules, max_depth=5):
        current = seq_names[:]
        for _ in range(max_depth):
            new_seq = []; i = 0; changed = False
            while i < len(current):
                matched = False
                for lhs, rhs in sorted(rules, key=lambda r: len(r[0]), reverse=True):
                    ln = [s.name for s in lhs]
                    if i+len(ln) <= len(current) and current[i:i+len(ln)] == ln:
                        new_seq.extend([s.name for s in rhs])
                        i += len(ln); matched = True; changed = True; break
                if not matched:
                    new_seq.append(current[i]); i += 1
            current = new_seq
            if not changed: break
        return current

    g = cls()
    if hasattr(g, 'set_level'):
        try: g.set_level(level)
        except: return None
    g.perform_action(ActionInput(id=GameAction.RESET), raw=True)

    if not all(hasattr(g, a) for a in ['ztgmtnnufb', 'cifzvbcuwqe', 'zvojhrjxxm', 'zdwrfusvmx']):
        return None

    n_elem = len(g.ztgmtnnufb)
    n_vals = len(g.zdwrfusvmx)

    seq_names = [s.name for s in g.zvojhrjxxm]
    targets = _parse_seq(seq_names, g.cifzvbcuwqe)

    if len(targets) != n_elem:
        return None

    rotations = []
    for ei in range(n_elem):
        gc = _cp.deepcopy(g)
        ci = gc.qvtymdcqear_index
        while ci != ei:
            gc.perform_action(ActionInput(id=GameAction.ACTION4), raw=True)
            ci = (ci + 1) % n_elem
        found = None
        for rot in range(n_vals):
            if gc.ztgmtnnufb[ei].name == targets[ei]:
                found = rot; break
            gc.perform_action(ActionInput(id=GameAction.ACTION1), raw=True)
        if found is None: return None
        rotations.append(found)

    path = []
    ci = g.qvtymdcqear_index
    for ei, rot in enumerate(rotations):
        while ci != ei:
            path.append((4, None))
            ci = (ci + 1) % n_elem
        for _ in range(rot):
            path.append((1, None))

    return path



# ============= TU93 SOLVER (ice sliding maze) =============
def _solve_tu93(cls, level_idx):
    """tu93: ice sliding maze on 6px cell grid. BFS on slide endpoints."""
    import numpy as np
    from collections import deque
    STEP = 3
    CELL = 6
    g = cls()
    if level_idx > 0:
        try: g.set_level(level_idx)
        except: return None
    g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    players = g.current_level.get_sprites_by_tag("albwnmiahg")
    exits = g.current_level.get_sprites_by_tag("xboyuzuyxv")
    board = g.current_level.get_sprites_by_tag("vhlesexlqd")[0]
    if not players or not exits:
        return None
    py, px = players[0].y - board.y, players[0].x - board.x
    ey, ex = exits[0].y - board.y, exits[0].x - board.x
    pixels = np.array(board.pixels)
    h, w = pixels.shape
    start = (py // CELL, px // CELL)
    goal = (ey // CELL, ex // CELL)
    def is_walkable(row, col):
        return 0 <= row < h and 0 <= col < w and pixels[row, col] == 2
    def can_move_to(from_r, from_c, dr, dc):
        fy, fx = from_r * CELL, from_c * CELL
        cy, cx = fy + dr * STEP, fx + dc * STEP
        return is_walkable(cy, cx)
    directions = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (cr, cc), path = queue.popleft()
        if (cr, cc) == goal:
            return [(a, None) for a in path]
        for action_id, (dr, dc) in directions.items():
            nr, nc = cr + dr, cc + dc
            if (nr, nc) not in visited and can_move_to(cr, cc, dr, dc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [action_id]))
    return None


class RewindAgent(Agent):
    MAX_ACTIONS: int = 500

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.levels = 0
        self.queue = []
        self.attempt = 0
        self.available = None
        self._game_cls = None
        self._solutions = {}
        self._solved_levels = set()
        self._total_presolve_time = 0
        self._load_and_presolve()
        logger.info(f'RewindAgent v24 init, game={self.game_id}, '
                    f'solutions={list(self._solutions.keys())}, '
                    f'presolve={self._total_presolve_time:.1f}s')

    def _load_and_presolve(self):
        """Load game source and pre-solve all levels."""
        env_dir = os.environ.get('ENVIRONMENTS_DIR', 'environment_files')
        short = self.game_id.split('-')[0]
        class_name = short[0].upper() + short[1:]

        candidates = [
            Path(env_dir) / short / f'{short}.py',
            Path(env_dir) / short / f'{class_name.lower()}.py',
            Path(env_dir) / f'{short}.py',
        ]

        for p in candidates:
            if p.exists():
                try:
                    spec = importlib.util.spec_from_file_location(f'g_{short}', str(p))
                    mod = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(mod)
                    self._game_cls = getattr(mod, class_name)
                    logger.info(f'Loaded game source: {p}')
                    break
                except Exception as e:
                    logger.warning(f'Failed to load {p}: {e}')

        if not self._game_cls:
            logger.warning(f'No game source for {short} in {env_dir}')
            return

        total_t0 = time.time()
        for level in range(20):
            if time.time() - total_t0 > 300:
                logger.info(f'Pre-solve budget exhausted at L{level}')
                break
            try:
                if not self._solve_level(level): break
            except Exception as e:
                logger.info(f'Pre-solve stopped at L{level}: {e}')
                break

        self._total_presolve_time = time.time() - total_t0
        logger.info(f'Pre-solved: {list(self._solutions.keys())} '
                    f'in {self._total_presolve_time:.1f}s')

    def _solve_level(self, level_idx):
        if self._game_cls is None: return False
        if level_idx in self._solved_levels:
            return level_idx in self._solutions
        self._solved_levels.add(level_idx)

        short = self.game_id.split('-')[0]

        sol = _solve_tr87(self._game_cls, level_idx)
        if sol:
            self._solutions[level_idx] = sol
            logger.info(f'L{level_idx} SOLVED by tr87: {len(sol)} actions')
            return True

        if short == 'dc22':
            sol = _solve_dc22(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by dc22 solver: {len(sol)} actions')
                return True

        if short == 'wa30':
            sol = _solve_wa30(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by wa30 solver: {len(sol)} actions')
                return True

        if short == 'r11l':
            sol = _solve_r11l(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by r11l solver: {len(sol)} actions')
                return True

        if short == 'su15':
            sol = _solve_su15(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by su15 solver: {len(sol)} actions')
                return True

        if short == 'tn36':
            sol = _solve_tn36(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by tn36 solver: {len(sol)} actions')
                return True

        if short == 'tu93':
            sol = _solve_tu93(self._game_cls, level_idx)
            if sol:
                self._solutions[level_idx] = sol
                logger.info(f'L{level_idx} SOLVED by tu93 solver: {len(sol)} actions')
                return True

                prioritize_clicks = short in ['r11l', 'su15', 'tn36']

        actions = _scan_actions(self._game_cls, level_idx, timeout=15,
                                prioritize_clicks=prioritize_clicks)
        if not actions: return False

        sol = _bfs(self._game_cls, actions, level_idx,
                   max_states=500000, timeout=45, game_id=short)
        if sol:
            self._solutions[level_idx] = sol
            logger.info(f'L{level_idx} SOLVED: {len(sol)} actions')
            return True
        return False

    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.queue = []
            self.attempt = 0
            return GameAction.RESET

        if latest_frame.levels_completed > self.levels:
            self.levels = latest_frame.levels_completed
            logger.info(f'LEVEL {self.levels} COMPLETE!')
            self.queue = []
            self.attempt = 0

        if self.available is None:
            self.available = latest_frame.available_actions or []

        if not self.queue and self.levels in self._solutions:
            self.queue = list(self._solutions[self.levels])
            logger.info(f'Loaded solution for L{self.levels}: '
                        f'{len(self.queue)} actions')

        if (not self.queue and self._game_cls and
                self.levels not in self._solutions):
            logger.info(f'Live solving L{self.levels}')
            if self._solve_level(self.levels):
                self.queue = list(self._solutions[self.levels])

        if self.queue:
            return self._execute_next()

        # No solution found — don't waste actions exploring
        return GameAction.RESET

    def _execute_next(self):
        act_id, data = self.queue.pop(0)
        if act_id == 6 and data:
            action = GameAction.ACTION6
            action.action_data.x = int(data['x'])
            action.action_data.y = int(data['y'])
            action.reasoning = f'v24 click ({data["y"]},{data["x"]})'
            return action
        action = GameAction.from_id(act_id)
        action.reasoning = f'v24 L{self.levels}'
        return action

    def cleanup(self, *a, **kw):
        if self._cleanup:
            logger.info(f'RewindAgent v24 final: {self.levels} levels, '
                        f'solutions={list(self._solutions.keys())}')
        super().cleanup(*a, **kw)


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy agent
    !cp /kaggle/working/rewind_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/rewind_agent.py

    # Patch __init__.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.rewind_agent import RewindAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"rewindagent\": RewindAgent,
}
""")

    # Write .env
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents/environment_files
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent rewindagent

In [ ]:
import os
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)